# Gear-2 trade chronology (signal vs fill)

**Track:** Glue / operator inspection (not live send, not collector).

Point this notebook at a **data dump** of Gear-2 journals plus nearby public L1 ticks. For each dual-leg `intent_id` you get:

1. L1 / tick series in a neighborhood around the event (OKX + Bybit).
2. Vertical markers for **signal**, **place/send**, **ack**, **fill**, and **dual_fill_complete** (later of the two fills).
3. Context: coin, `spread_side`, qty, venues, and journal `fill_source` when present.

## How to run against a VPS dump

Copy (or rclone) onto the machine that runs Jupyter, then set paths in the config cell:

| Role | Typical VPS path | Notebook knobs |
|------|------------------|----------------|
| Public legs journal | `/data/bbot-gear2/journal/event_date=*/legs.jsonl` | `DATA_ROOT` → parent of `journal/` (e.g. `/data/bbot-gear2` or a copied tree) |
| Private events (optional) | `/data/bbot-gear2/private/journal/event_date=*/events.jsonl` | same `DATA_ROOT` if layout is `…/private/journal/…` |
| Lean L1 hive | `/data/live/base_coin=<COIN>/event_date=*/…parquet` | `TICKS_ROOT=/data/live` (or a copied hive) |
| Lean L1 compacted | `/data/compacted/spread_*.parquet` | `TICKS_ROOT=/data/compacted` |

```bash
# example: local dump next to the notebook
DATA_ROOT=./dumps/bbot-gear2
TICKS_ROOT=./dumps/live   # or ./dumps/compacted
INTENT_ID=<uuid-from-legs.jsonl>
```

Hermetic smoke (no VPS disk):

```bash
python3 -m pytest tests/test_gear2_trade_chronology.py -q
```

Or leave `USE_FIXTURE=True` below.

## Caveats

- **`fill_source=l1_at_send`** (when present in `legs.jsonl`) means the fill timestamp/price came from **public L1 at send time**, not a venue-reported match. The chart **labels** that source; it does **not** pretend it is an exchange fill.
- Public stub/gear2 journal fill is also L1-at-`Trade_Lat` semantics in `bbot.journal.v0` — still not a private match.
- Private `events.jsonl` markers (`request_sent` / `ack_received` / `terminal_update`) use wall-clock `event_ts_utc` and are drawn dashed when present.
- Default neighborhood pad is **±3 seconds** around signal/fill anchors (`PAD_MS`). Widen if ticks look sparse.
- Time axis is **UTC** (ms precision from journal fields). Set `OFFSET_MS` to shift the axis (e.g. relative to signal) without changing stored timestamps.
- Do not point this notebook at secrets, `.env`, or `/etc/spread`.

Helpers: [`research/gear2_trade_chronology.py`](gear2_trade_chronology.py).

In [ ]:
from pathlib import Path
import sys

REPO = Path.cwd()
if not (REPO / "research" / "gear2_trade_chronology.py").is_file():
    # notebook opened from research/
    if (REPO.parent / "research" / "gear2_trade_chronology.py").is_file():
        REPO = REPO.parent
sys.path.insert(0, str(REPO))

from research.gear2_trade_chronology import (
    DEFAULT_PAD_MS,
    FIXTURE_INTENT_ID,
    build_chronology,
    fixture_root,
    list_intent_ids,
    make_chronology_figure,
    summarize_intent,
)

# --- operator entry: point me at data_root / intent_id ---
USE_FIXTURE = True  # set False for a real dump
DATA_ROOT = fixture_root(REPO) if USE_FIXTURE else Path("/data/bbot-gear2")
TICKS_ROOT = (fixture_root(REPO) / "ticks") if USE_FIXTURE else Path("/data/live")
INTENT_ID = FIXTURE_INTENT_ID if USE_FIXTURE else ""  # fill from list_intent_ids()
PAD_MS = DEFAULT_PAD_MS  # neighborhood pad around signal/fill (ms)
OFFSET_MS = 0  # e.g. signal_ts_ms to plot relative time

print("DATA_ROOT", DATA_ROOT)
print("TICKS_ROOT", TICKS_ROOT)
print("available intents", list_intent_ids(DATA_ROOT)[:20])

In [ ]:
if not INTENT_ID:
    ids = list_intent_ids(DATA_ROOT)
    if not ids:
        raise SystemExit(f"no intents under {DATA_ROOT}/journal/event_date=*/legs.jsonl")
    INTENT_ID = ids[0]
    print("using first intent", INTENT_ID)

plot = build_chronology(
    DATA_ROOT,
    INTENT_ID,
    ticks_root=TICKS_ROOT,
    pad_ms=PAD_MS,
    offset_ms=OFFSET_MS,
)
summary = summarize_intent(plot)
summary

In [ ]:
fig = make_chronology_figure(plot)
fig.show()

# Optional: write HTML next to the dump for offline viewing
# out = Path(DATA_ROOT) / f"chronology_{INTENT_ID}.html"
# fig.write_html(out)
# print("wrote", out)

## Reading the figure

- Top panel: `spread_long` / `spread_short` from public L1 in the window.
- Middle / bottom: OKX and Bybit mid — dual-leg venues side by side.
- Solid markers from public `legs.jsonl`; dashed from private `events.jsonl` when `dual_leg_id` matches `intent_id`.
- `dual_fill_complete` = max(okx fill, bybit fill). Title shows `signal→dual_fill` ms when both fills exist.
- If title mentions `fill_source=l1_at_send`, treat fill markers as **L1 proxy**, not venue match time.